# Phase 1: Extract Financial Data from SEC XBRL API

## Overview

This notebook demonstrates all 3 SEC XBRL API endpoints:
1. **Endpoint 1:** Company Facts - All financial data for a company
2. **Endpoint 2:** Company Concept - Specific metric over time
3. **Endpoint 3:** Submissions - Filing metadata and URLs

Uses orchestrator class for simplified access.

In [ ]:
import sys
sys.path.insert(0, '/Users/prajwalchambenandeeshappa/Github_Repos/Stocks_Earnings_Intelligence_Agent-Text2SQL/learning')

import logging
from sec_api_orchestrator import FinancialDataPipeline
from sec_api_endpoint1 import get_company_facts
from sec_api_endpoint2 import get_concept_over_time
from sec_api_endpoint3 import get_10q_filings_metadata

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

TICKERS = ["MSFT", "AAPL", "GOOGL"]
KEY_METRICS = ["Revenues", "NetIncomeLoss", "OperatingIncomeLoss", "Assets", "Liabilities", "StockholdersEquity"]

print("✅ Imports successful")
print(f"   Companies: {', '.join(TICKERS)}")
print(f"   Metrics: {', '.join(KEY_METRICS[:3])}...")

---
## Method 1: Using Orchestrator (Recommended)

Simple interface for all 3 endpoints combined

In [ ]:
print("\n" + "="*70)
print("METHOD 1: Using FinancialDataPipeline Orchestrator")
print("="*70)

# Create pipeline for one company
pipeline = FinancialDataPipeline("AAPL")

# Get complete data (all 3 endpoints)
complete_data = pipeline.get_complete_data(
    metrics=["Revenues", "NetIncomeLoss"],
    facts_limit=2,
    filings_limit=3
)

print(f"\n✅ Complete data retrieved:")
print(f"   Facts: {len(complete_data['facts'])} concepts")
print(f"   Metrics: {list(complete_data['metrics'].keys())}")
print(f"   Filings: {len(complete_data['filings'])} recent 10-Q/10-K")

---
## Method 2: Individual Endpoints

For more control, use endpoints directly

In [ ]:
print("\n" + "="*70)
print("ENDPOINT 1: Company Facts (All Financial Data)")
print("="*70)

facts = get_company_facts("MSFT", limit_per_concept=2)
print(f"\n✅ Retrieved {len(facts)} financial concepts for MSFT")
print(f"\nSample concepts:")
for concept_name in list(facts.keys())[:5]:
    data_points = facts[concept_name]
    print(f"   {concept_name}: {len(data_points)} data points")

In [ ]:
print("\n" + "="*70)
print("ENDPOINT 2: Company Concept (Metric Over Time)")
print("="*70)

revenues = get_concept_over_time("GOOGL", concept="Revenues", limit=5)
print(f"\n✅ Retrieved {len(revenues)} revenue data points for GOOGL")
print(f"\nMost recent entries:")
for i, record in enumerate(revenues[:3], 1):
    value_b = record['value'] / 1000000000
    print(f"   {i}. {record['period_end']} ({record['form']}): ${value_b:.1f}B")

In [ ]:
print("\n" + "="*70)
print("ENDPOINT 3: Submissions (Filing Metadata)")
print("="*70)

filings = get_10q_filings_metadata("AAPL", limit=3)
print(f"\n✅ Retrieved {len(filings)} recent 10-Q/10-K filings for AAPL")
print(f"\nFilings:")
for i, filing in enumerate(filings, 1):
    print(f"   {i}. {filing['filing_date']} ({filing['form']})")
    print(f"      Accession: {filing['accession_number']}")
    print(f"      URL: {filing['filing_url'][:60]}...")

---
## Fetch All Data for All Companies

Prepare data for Phase 2 loading

In [ ]:
print("\n" + "="*70)
print("COLLECTING DATA FOR PHASE 2")
print("="*70)

all_metrics = []
all_filings = []

for ticker in TICKERS:
    print(f"\n📊 {ticker}...")
    pipeline = FinancialDataPipeline(ticker)
    
    # Get metrics
    for metric in KEY_METRICS:
        metric_data = pipeline.get_metric(metric, limit=10)
        all_metrics.extend(metric_data)
    
    # Get filings
    filings_data = pipeline.get_filings(limit=5)
    all_filings.extend(filings_data)

print(f"\n✅ Data collection complete:")
print(f"   Total metrics: {len(all_metrics)} records")
print(f"   Total filings: {len(all_filings)} records")

In [ ]:
# Show sample data
print("\n" + "="*70)
print("SAMPLE DATA")
print("="*70)

print("\n📊 Sample metric record:")
sample = all_metrics[0]
for key, value in sample.items():
    print(f"   {key}: {value}")

print("\n📄 Sample filing record:")
sample = all_filings[0]
for key, value in sample.items():
    if key == 'filing_url':
        print(f"   {key}: {value[:60]}...")
    else:
        print(f"   {key}: {value}")

---
## Summary

In [ ]:
print("\n" + "="*70)
print("PHASE 1 COMPLETE ✅")
print("="*70)

print(f"""
✅ SEC XBRL API Data Extracted:

📊 Financial Metrics:
   • {len(all_metrics)} records
   • {len(TICKERS)} companies
   • {len(KEY_METRICS)} metrics
   • Data ready for dlt loading

📄 Filing Metadata:
   • {len(all_filings)} records
   • Filing dates and URLs
   • Ready for MD&A extraction

🚀 Next: Phase 2 - Load into DuckDB
   See: PHASE_2_COMPLETE_DLT_DUCKDB.ipynb

📚 Key Learnings:
   ✅ SEC XBRL API is official source
   ✅ 3 endpoints serve different purposes
   ✅ Orchestrator simplifies access
   ✅ Data is standardized (XBRL tags)
   ✅ Scales to thousands of companies
""")

print("="*70)